# ZelusBench: Structural Attention

**How does dependency graph topology affect accuracy?**

Varies DAG structure while randomizing ALL background conditions
(chain depth, distractors, transforms, dimensionality, point types).

- Topologies: LINEAR, BRANCHING, MERGING, DIAMOND
- 10 seeds per topology = 40 scenarios, ~120 queries

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import re
import time

from zelusbench.scenarios.config import ScenarioConfig, DAGStructure
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.parser import parse_model_response
from zelusbench.evaluation.scorer import score_query


def score_response(response_text, scenario):
    query_dicts = [q.to_dict() for q in scenario.queries]
    parts = re.split(r'\[Query\s+q_\d+\]', response_text)
    if len(parts) > 1:
        parts = parts[1:]
    if len(parts) != len(query_dicts):
        parts = [response_text] * len(query_dicts)
    return [score_query(qd, parse_model_response(rp, qd))
            for qd, rp in zip(query_dicts, parts)]


TOPOLOGIES = [
    ("LINEAR", DAGStructure.LINEAR),
    ("BRANCHING", DAGStructure.BRANCHING),
    ("MERGING", DAGStructure.MERGING),
    ("DIAMOND", DAGStructure.DIAMOND),
]
SEEDS = 10
TOTAL = len(TOPOLOGIES) * SEEDS

print(f"ZelusBench Structural Attention")
print(f"Topologies: {[n for n, _ in TOPOLOGIES]} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
@kbench.task(name="zelusbench_structural_attention")
def zelusbench_structural_attention(llm) -> tuple[float, float]:
    """Structural attention: accuracy by DAG topology (randomized background).
    Returns (mean_accuracy, std_dev)."""

    _start = time.time()
    print(f"Running {TOTAL} scenarios across {len(TOPOLOGIES)} topologies...")
    print("=" * 60)

    all_scores = []
    topo_scores = {}
    scenario_num = 0

    for idx, (name, structure) in enumerate(TOPOLOGIES):
        topo_scores[name] = []
        for seed in range(SEEDS):
            scenario_num += 1
            unique_seed = idx * 1000 + seed
            rng = random.Random(unique_seed)
            cfg = ScenarioConfig.randomize_except(rng, pinned={
                "dag_structure": structure,
                "num_queries": 3,
                "seed": unique_seed,
            })
            gen = ScenarioGenerator(cfg)
            scenario = gen.generate(f"structure_{name}_s{seed}")

            response = llm.prompt(scenario.prompt)
            scores = score_response(response, scenario)
            all_scores.extend(scores)

            avg = float(np.mean([s.score for s in scores]))
            topo_scores[name].append(avg)

            q_depths = [s.chain_depth for s in scores]
            tiers = [s.tier.name for s in scores]
            bg = f"dim={cfg.dim} depth={cfg.min_chain_depth}-{cfg.max_chain_depth} dist={cfg.distractor_level.name} tx={cfg.transform_density.name}"
            print(f"  [{scenario_num}/{TOTAL}] topology={name} seed={seed} "
                  f"avg={avg:.2f} q_depths={q_depths} tiers={tiers} | {bg}")

        topo_avg = float(np.mean(topo_scores[name]))
        print(f"  >> {name} mean: {topo_avg:.3f}")

    print("\n" + "=" * 60)
    print("RESULTS BY TOPOLOGY:")
    for name, _ in TOPOLOGIES:
        avg = float(np.mean(topo_scores[name]))
        print(f"  {name:12s}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg >= 0,
            expectation=f"{name}: model should produce valid answers (got {avg:.3f})"
        )

    overall = float(np.mean([s.score for s in all_scores]))
    std = float(np.std([s.score for s in all_scores]))
    elapsed = time.time() - _start

    print(f"\nOverall: {overall:.3f} +/- {std:.3f}")
    print(f"Total queries: {len(all_scores)}")
    print(f"Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

    return overall, std


zelusbench_structural_attention.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_structural_attention